# **Start Section:**

In [1]:
# FIX K1: same notebook runs on Google Colab (Drive) or Kaggle (internet on).
# On Kaggle the working dir is /kaggle/working, so the original "./drive/MyDrive/..." paths
# below are reused unchanged; the dataset is copied there as train.csv / test.csv.
# The IBUG cells later change directory (%cd ibug, %cd .., %cd ..) and then use
# "./content/drive/..." and "/content/drive/..." (Colab layout); on Kaggle both are linked
# to /kaggle/working so every Excel file lands in the same output folder.
import os, glob, shutil
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_KAGGLE:
    _data_dir = "./drive/MyDrive/Regression/pile_uncertainty_analysis/data"
    os.makedirs(_data_dir, exist_ok=True)
    shutil.copy(glob.glob("/kaggle/input/**/d08_a.csv", recursive=True)[0], f"{_data_dir}/train.csv")  # pile train
    shutil.copy(glob.glob("/kaggle/input/**/d08_b.csv", recursive=True)[0], f"{_data_dir}/test.csv")   # pile test
    for _link in ["/content", "/kaggle/content"]:
        if not os.path.exists(_link):
            os.symlink("/kaggle/working", _link)
    print("Kaggle: pile data copied to", _data_dir)
else:
    from google.colab import drive
    drive.mount('/content/drive')


Kaggle: pile data copied to ./drive/MyDrive/Regression/pile_uncertainty_analysis/data


In [2]:
# FIX K1: the treeffuser fork pins numpy<2 / scikit-learn==1.5.0 / lightgbm==4.3.0, which cannot be
# installed on Kaggle's Python 3.13. On Kaggle install the same code without those pins.
if IN_KAGGLE:
    !pip install -q --no-deps git+https://github.com/DaneshSelwal/treeffuser.git
    !pip install -q jaxtyping einops ml-collections
else:
    !pip install git+https://github.com/DaneshSelwal/treeffuser.git


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
treeffuser 0.1.3 requires jaxtyping<0.3.0,>=0.2.19, but you have jaxtyping 0.3.11 which is incompatible.
treeffuser 0.1.3 requires lightgbm==4.3.0, but you have lightgbm 4.6.0 which is incompatible.
treeffuser 0.1.3 requires ml-collections<0.2.0,>=0.1.1, but you have ml-collections 1.1.0 which is incompatible.
treeffuser 0.1.3 requires numpy<2.0,>=1.24, but you have numpy 2.1.3 which is incompatible.
treeffuser 0.1.3 requires scikit-learn==1.5.0, but you have scikit-learn 1.6.1 which is incompatible.


In [3]:
# FIX K1: original Colab install block kept as-is (runs only on Colab).
# On Kaggle: install only the packages the Kaggle image lacks, pinned to the Colab versions,
# and do NOT restart the kernel with os._exit(0).
if IN_KAGGLE:
    !pip install -q optuna==4.6.0 optunahub==0.4.0 cmaes==0.12.0 xgboost==3.1.2 catboost==1.2.8 gpboost==1.6.1 ngboost==0.5.8 pgbm==2.2.0 pytorch-tabnet2==4.5.3 interpret==0.7.4 lime==0.2.0.1 mapie==0.6.0 puncc==0.8.0 properscoring==0.1 XlsxWriter==3.2.9
else:
    import os
    os.environ["PIP_CONSTRAINT"] = "/tmp/numpy_constraint.txt"
    !echo "numpy==1.26.4" > /tmp/numpy_constraint.txt

    !pip uninstall -y torch torchaudio torchvision \
        torchao torchcodec torchdata torchtune torchsummary -q 2>/dev/null

    !pip uninstall -y tensorflow tensorflow-text tensorflow-hub tf-keras \
        tensorflow_decision_forests tensorflow-probability \
        tensorflow-Datasets tensorflow-metadata -q 2>/dev/null

    !pip uninstall -y numpy scikit-learn shap xgboost lightgbm dask \
        seaborn plotly openpyxl Cython catboost interpret lime -q 2>/dev/null

    !pip install numpy==1.26.4 -q
    !pip install scikit-learn==1.6.1 -q
    !pip install torch==2.9.0 -q

    !pip install lightgbm==4.6.0 -q
    !pip install xgboost==3.1.2 -q
    !pip install catboost==1.2.8 -q
    !pip install gpboost==1.6.1 -q
    !pip install ngboost==0.5.8 -q
    !pip install pgbm==2.2.0 -q
    !pip install pytorch-tabnet2==4.5.3 -q

    !pip install bayesian-optimization==3.2.0 -q
    !pip install optuna==4.6.0 -q
    !pip install optunahub==0.4.0 -q
    !pip install cmaes==0.12.0 -q

    !pip install shap==0.44.0 -q
    !pip install lime==0.2.0.1 -q
    !pip install interpret==0.7.4 -q

    !pip install mapie==0.6.0 -q
    !pip install puncc==0.8.0 -q
    !pip install skorch==1.3.1 -q
    !pip install properscoring==0.1 -q

    !pip install dask[dataframe]==2025.12.0 -q
    !pip install cython==3.0.12 -q
    !pip install seaborn==0.13.2 -q
    !pip install plotly==5.24.1 -q
    !pip install kaleido==1.2.0 -q
    !pip install openpyxl==3.1.5 -q
    !pip install XlsxWriter==3.2.9 -q
    !pip install cp==2020.12.3 -q

    !pip install numpy==1.26.4 --force-reinstall --no-deps -q

    os._exit(0)


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


# **Imports**

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ngboost
from scipy.stats import randint
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from gpboost import GPBoostRegressor
from ngboost import NGBRegressor
import optuna
import optunahub
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from interpret import show
from interpret.blackbox import LimeTabular, ShapKernel
from optuna.samplers import RandomSampler
import random
import time
from ngboost.distns import Normal
from ngboost.scores import LogScore
from scipy.stats import norm
from interpret import set_visualize_provider
from interpret.provider import InlineProvider
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show
import plotly.express as px
from io import BytesIO
from openpyxl import Workbook, load_workbook
import os
import properscoring as ps
import io
from openpyxl.drawing.image import Image as openpyxlImage
import warnings
import xlsxwriter
from pgbm.sklearn import HistGradientBoostingRegressor
import torch
from pgbm.torch import PGBM
import plotly.graph_objects as go
warnings.filterwarnings('ignore')
import pickle
import json
from mapie.subsample import Subsample
from mapie.regression import MapieRegressor
from deel.puncc.metrics import regression_sharpness, regression_mean_coverage
from deel.puncc.api.prediction import BasePredictor, DualPredictor
from deel.puncc.regression import SplitCP, CVPlus, CQR
from deel.puncc.plotting import plot_prediction_intervals
from sklearn.model_selection import train_test_split
from typing_extensions import TypedDict
from typing import Union
from mapie.metrics import regression_coverage_score
from sklearn.model_selection import KFold
from PIL import Image as PImage
from pathlib import Path
from scipy.stats import norm, kstest
from pytorch_tabnet import TabNetRegressor
from treeffuser import Treeffuser, Samples
from treeffuser._score_models import TreeBasedScoreModel

[1/2] c++ -MMD -MF splitgain_cpu.o.d -DTORCH_EXTENSION_NAME=split_decision -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.13/dist-packages/torch/include -isystem /usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/include/python3.13 -fPIC -std=c++17 -c /usr/local/lib/python3.13/dist-packages/pgbm/torch/splitgain_cpu.cpp -o splitgain_cpu.o 


[2/2] c++ splitgain_cpu.o -shared -L/usr/local/lib/python3.13/dist-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o split_decision.so


I0000 00:00:1795281084.139073      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1795281101.412420      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [5]:
# Go to find & replace button and replace (pile_uncertainty_analysis) with your folder name. Rename your train and test Dataset as train.csv and test.csv.
# Modify the names of the feature in the below cell.
# Replace (str) with actual data label name.

In [6]:
feature_names = ['F1', 'F2', 'Zv1', 'Zv2', 'phi', 'Th', 'L']

In [7]:
train_data_path = "./drive/MyDrive/Regression/pile_uncertainty_analysis/data/train.csv"
test_data_path = "./drive/MyDrive/Regression/pile_uncertainty_analysis/data/test.csv"
train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)
print("Training data loaded successfully.")
print("Test data loaded successfully.")

Training data loaded successfully.
Test data loaded successfully.


In [8]:
print("\nShape of training data:", train_data.shape)
print("First 5 rows of training data:\n", train_data.head(5))
print("\nShape of test data:", test_data.shape)
print("First 5 rows of test data:\n", test_data.head(5))


Shape of training data: (81, 8)
First 5 rows of training data:
      F1    F2   Zv1   Zv2  phi     Th   L  Total
0  4.80  1.21  4.90 -1.87  0.5  0.125  32   5.55
1  4.34  1.09  4.36 -1.95  0.5  0.125  45   6.07
2  4.71  0.81  5.10 -1.49  0.5  0.125  42   6.06
3  5.55  1.04  5.30 -1.46  0.5  0.125  38   5.94
4  3.08  0.95  2.82 -1.60  0.4  0.097  36   3.88

Shape of test data: (24, 8)
First 5 rows of test data:
      F1    F2   Zv1   Zv2  phi     Th   L  Total
0  4.69  1.06  4.70 -1.82  0.5  0.125  42   6.12
1  3.52  0.64  3.55 -1.03  0.4  0.097  45   4.26
2  3.70  0.49  3.43 -1.13  0.5  0.100  44   4.78
3  4.17  1.45  4.44 -0.21  0.5  0.125  16   5.78
4  4.33  2.04  4.75 -1.66  0.5  0.125  32   5.84


In [9]:
X_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]
X_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]
x_test= X_test
print("\nShape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)


Shape of X_train: (81, 7)
Shape of y_train: (81,)
Shape of X_test: (24, 7)
Shape of y_test: (24,)


In [10]:
# Apply z-score normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Print the first five rows of the normalized data
print("\nFirst five rows of normalized X_train:")
print(X_train[:5])

print("\nFirst five rows of normalized X_test:")
print(X_test[:5])


First five rows of normalized X_train:
[[ 0.006145   -0.12114279  0.17814132 -0.04043083 -0.57071458  0.40496686
  -0.73776147]
 [-0.37545934 -0.27898568 -0.27593219 -0.10402107 -0.57071458  0.40496686
   0.23089542]
 [-0.06851672 -0.64728574  0.3463167   0.2616228  -0.57071458  0.40496686
   0.00735922]
 [ 0.62832598 -0.34475355  0.51449207  0.28546914 -0.57071458  0.40496686
  -0.29068906]
 [-1.42072338 -0.46313571 -1.57088258  0.17418622 -2.53785844 -2.27277321
  -0.4397132 ]]

First five rows of normalized X_test:
[[-8.51082130e-02 -3.18446397e-01  9.96594816e-03 -6.86931582e-04
  -5.70714577e-01  4.04966863e-01  7.35921670e-03]
 [-1.05571054e+00 -8.70896496e-01 -9.57042459e-01  6.27266667e-01
  -2.53785844e+00 -2.27277321e+00  2.30895424e-01]
 [-9.06387106e-01 -1.06820010e+00 -1.05794768e+00  5.47778870e-01
  -5.70714577e-01 -1.98587249e+00  1.56383355e-01]
 [-5.16487026e-01  1.94542980e-01 -2.08662040e-01  1.27906661e+00
  -5.70714577e-01  4.04966863e-01 -1.92995458e+00]
 [-3.83

# **Functions:**

In [11]:
# Define the model classes
model_classes = {
    'Random Forest': RandomForestRegressor,
    'Gradient Boosting': GradientBoostingRegressor,
    'XGBoost': XGBRegressor,
    'LightGBM': LGBMRegressor,
    'GPBoost': GPBoostRegressor,
    'CatBoost': CatBoostRegressor,
    'HistGradientBoosting': HistGradientBoostingRegressor,
    'TabNet': TabNetRegressor,
    'NGBoost': NGBRegressor
}

def _ensure_excel_file(path):
    import os
    import pandas as pd
    if not os.path.exists(path):
        parent = os.path.dirname(path)
        if parent:
            os.makedirs(parent, exist_ok=True)
        pd.DataFrame().to_excel(path)
    return path

In [12]:
def get_best_model_params(results, model_name):
    # Map model names to dictionary keys, assuming keys are strings like 'XGBoost' and not objects
    model_keys = {
        'LightGBM': 'LightGBM',
        'XGBoost': 'XGBoost',
        'GPBoost': 'GPBoost',
        'NGBoost': 'NGBoost',
        'GradientBoosting': 'Gradient Boosting',
        'HGBR' : 'HistGradientBoosting',
        'CatBoost' : 'CatBoost',
        'PGBM' : 'PGBM'
    }

    # Ensure the requested model name is valid
    if model_name not in model_keys:
        raise ValueError(f"Model name '{model_name}' is not recognized. Available models are: {list(model_keys.keys())}")

    # Filter out entries for the specified model
    model_entries = {key: value for key, value in results.items() if key[0] == model_keys[model_name]}

    # Find the entry with the best (lowest) 'best_score'
    best_entry_key, best_entry_value = min(model_entries.items(), key=lambda item: item[1]['best_score'])

    # Return the best hyperparameters
    return best_entry_value['best_params']

In [13]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def generate_and_save_plots(predictions_df, y_test, excel_path):
    # 1. Predicted vs. Actual Plot
    plt.figure(figsize=(12, 8))
    sns.scatterplot(x='Actual', y='Mean', data=predictions_df)
    plt.plot([predictions_df['Actual'].min(), predictions_df['Actual'].max()],
             [predictions_df['Actual'].min(), predictions_df['Actual'].max()],
             color='red', linestyle='--')
    plt.title('Predicted str vs. Actual str')
    plt.xlabel('Actual str')
    plt.ylabel('Predicted Mean str')
    plt.tight_layout()
    predicted_vs_actual_path = 'predicted_vs_actual.png'
    _ensure_parent_dir(predicted_vs_actual_path)
    plt.savefig(_ensure_parent_dir(predicted_vs_actual_path))
    plt.close()

    # 2. Uncertainty Bands
    plt.figure(figsize=(12, 8))
    plt.errorbar(x=range(len(predictions_df)), y=predictions_df['Mean'],
                 yerr=predictions_df['StdDev'], fmt='o', markersize=4, ecolor='red', elinewidth=1, capsize=5, alpha=0.7)
    plt.plot(range(len(predictions_df)), predictions_df['Mean'], color='black', linewidth=0.5, alpha=0.7)
    plt.title('Predicted Mean with Uncertainty Bands', fontsize=16)
    plt.xlabel('Sample Number', fontsize=14)
    plt.ylabel('Predicted Mean str')
    plt.tight_layout()
    uncertainty_bands_path = 'uncertainty_bands.png'
    _ensure_parent_dir(uncertainty_bands_path)
    plt.savefig(_ensure_parent_dir(uncertainty_bands_path))
    plt.close()

    # 3. Histogram of Standard Deviations
    plt.figure(figsize=(12, 8))
    sns.histplot(predictions_df['StdDev'], bins=20, kde=True)
    plt.title('Distribution of Predicted Standard Deviations')
    plt.xlabel('Standard Deviation')
    plt.ylabel('Frequency')
    plt.tight_layout()
    histogram_stddev_path = 'histogram_stddev.png'
    _ensure_parent_dir(histogram_stddev_path)
    plt.savefig(_ensure_parent_dir(histogram_stddev_path))
    plt.close()

    # 4. Residual Plot
    residuals = predictions_df['Actual'] - predictions_df['Mean']
    plt.figure(figsize=(12, 8))
    sns.scatterplot(x=predictions_df['Mean'], y=residuals)
    plt.axhline(0, color='red', linestyle='--')
    plt.title('Residual Plot')
    plt.xlabel('Predicted Mean str')
    plt.ylabel('Residuals')
    plt.tight_layout()
    residual_plot_path = 'residual_plot.png'
    _ensure_parent_dir(residual_plot_path)
    plt.savefig(_ensure_parent_dir(residual_plot_path))
    plt.close()

    # Plot the standard deviation of the predictions
    plt.figure(figsize=(12, 6))
    plt.plot(
        predictions_df['StdDev'],
        label='Standard Deviation',
        color='green',
        marker='o',
        linestyle='dashed'
    )
    plt.xlabel('Sample Number')
    plt.ylabel('Standard Deviation')
    plt.title('Standard Deviation of Predictions')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    plt.tight_layout()
    stddev_plot_path = 'stddev_plot.png'
    _ensure_parent_dir(stddev_plot_path)
    plt.savefig(_ensure_parent_dir(stddev_plot_path))
    plt.close()

    # Plot the actual vs predicted mean with confidence intervals
    plt.figure(figsize=(12, 6))
    plt.plot(
        y_test,
        label='Actual str',
        color='black',
        marker='o',
        linestyle='dashed',
        linewidth=1,
        markersize=4
    )
    plt.plot(
        predictions_df['Mean'],
        label='Predicted Mean str',
        color='red',
        marker='x',
        linestyle='dashed',
        linewidth=1,
        markersize=4
    )
    confidence_interval = 1.96 * predictions_df['StdDev']
    plt.fill_between(
        np.arange(len(predictions_df)),
        predictions_df['Mean'] - confidence_interval,
        predictions_df['Mean'] + confidence_interval,
        color='blue',
        alpha=0.2,
        label='95% Confidence Interval'
    )
    plt.xlabel('Sample Number')
    plt.ylabel(' str')
    plt.title('Predictions with Confidence Intervals')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    plt.tight_layout()
    prediction_plot_path = 'prediction_plot.png'
    _ensure_parent_dir(prediction_plot_path)
    plt.savefig(_ensure_parent_dir(prediction_plot_path))
    plt.close()

    # Create a Pandas Excel writer using XlsxWriter as the engine.
    writer = pd.ExcelWriter(_ensure_parent_dir(excel_path), engine='xlsxwriter')

    # Write the dataframe to a worksheet.
    writer
    predictions_df.to_excel(writer, sheet_name='Predictions', index=False)

    # Access the XlsxWriter workbook and worksheet objects from the writer object.
    workbook = writer.book

    # Insert the plots into new worksheets.
    worksheet_predicted_vs_actual = workbook.add_worksheet('Predicted vs Actual')
    worksheet_predicted_vs_actual.insert_image('A1', predicted_vs_actual_path)

    worksheet_uncertainty_bands = workbook.add_worksheet('Uncertainty Bands')
    worksheet_uncertainty_bands.insert_image('A1', uncertainty_bands_path)

    worksheet_histogram_stddev = workbook.add_worksheet('Histogram StdDev')
    worksheet_histogram_stddev.insert_image('A1', histogram_stddev_path)

    worksheet_residual_plot = workbook.add_worksheet('Residual Plot')
    worksheet_residual_plot.insert_image('A1', residual_plot_path)

    worksheet_stddev = workbook.add_worksheet('StdDev Plot')
    worksheet_stddev.insert_image('A1', stddev_plot_path)

    worksheet_prediction = workbook.add_worksheet('Predictions Plot')
    worksheet_prediction.insert_image('A1', prediction_plot_path)

    # Close the Pandas Excel writer and output the Excel file.
    writer.close()
    print(f"Excel file saved at {excel_path}")

    # Optionally, remove the image files now that they're embedded in the Excel file
    if os.path.exists(str(predicted_vs_actual_path)): os.remove(str(predicted_vs_actual_path))
    if os.path.exists(str(uncertainty_bands_path)): os.remove(str(uncertainty_bands_path))
    if os.path.exists(str(histogram_stddev_path)): os.remove(str(histogram_stddev_path))
    if os.path.exists(str(residual_plot_path)): os.remove(str(residual_plot_path))
    if os.path.exists(str(stddev_plot_path)): os.remove(str(stddev_plot_path))
    if os.path.exists(str(prediction_plot_path)): os.remove(str(prediction_plot_path))

In [14]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def generate_and_save_crps_decomposition(
    predictions_df,
    y_true,
    excel_path,
    model_name="Model",
    n_bins=10
):

    # -------------------------------
    # Input handling
    # -------------------------------
    mu_pred = predictions_df['Mean'].values
    sigma_pred = predictions_df['StdDev'].values

    if 'Actual' in predictions_df.columns:
        y_true = predictions_df['Actual'].values
    else:
        y_true = np.asarray(y_true).ravel()

    y_true = np.asarray(y_true).ravel()

    # -------------------------------
    # Total CRPS
    # -------------------------------
    crps_values = ps.crps_gaussian(
        y_true, mu=mu_pred, sig=sigma_pred
    )
    total_crps = np.mean(crps_values)

    # -------------------------------
    # FIX P1: Reliability / Resolution / Uncertainty from the Hersbach (2000) decomposition,
    # "Decomposition of the CRPS for ensemble prediction systems", Weather and Forecasting 15.
    # Was: Reliability = mean of the model's own CRPS over 10 groups (= total CRPS), so
    # Resolution always came out = Uncertainty and the split carried no information.
    # Each predicted Gaussian is represented by M members at its quantile levels (k-0.5)/M.
    #   CRPS_ens = Reliability + CRPS_pot,  CRPS_pot = Uncertainty - Resolution
    # -------------------------------
    M = 100
    sig_safe = np.clip(sigma_pred, 1e-12, None)
    levels = (np.arange(1, M + 1) - 0.5) / M
    ens = np.sort(norm.ppf(levels[None, :], loc=mu_pred[:, None], scale=sig_safe[:, None]), axis=1)  # (n, M)
    yy = y_true[:, None]
    n_obs = len(y_true)

    # alpha_i, beta_i for the inner bins [x_i, x_{i+1}], i = 1..M-1 (Hersbach eq. 26)
    lo, hi = ens[:, :-1], ens[:, 1:]
    alpha_in = np.where(yy > hi, hi - lo, np.where(yy > lo, yy - lo, 0.0))
    beta_in = np.where(yy < lo, hi - lo, np.where(yy < hi, hi - yy, 0.0))
    alpha_bar = np.zeros(M + 1)
    beta_bar = np.zeros(M + 1)
    alpha_bar[1:M] = alpha_in.mean(axis=0)
    beta_bar[1:M] = beta_in.mean(axis=0)
    # outliers (Hersbach eq. 27): below the lowest / above the highest member
    beta_bar[0] = np.mean(np.where(y_true < ens[:, 0], ens[:, 0] - y_true, 0.0))
    alpha_bar[M] = np.mean(np.where(y_true > ens[:, -1], y_true - ens[:, -1], 0.0))

    p = np.arange(M + 1) / M
    g = alpha_bar + beta_bar
    o = np.divide(beta_bar, g, out=np.zeros_like(g), where=g > 0)
    # outlier bins: o_0 = freq(y < x_1), o_M = 1 - freq(y > x_M) (Hersbach eq. 33)
    o[0] = np.mean(y_true < ens[:, 0])
    o[M] = 1.0 - np.mean(y_true > ens[:, -1])
    g[0] = beta_bar[0] / o[0] if o[0] > 0 else 0.0
    g[M] = alpha_bar[M] / (1.0 - o[M]) if o[M] < 1 else 0.0

    reliability = float(np.sum(g * (o - p) ** 2))
    crps_pot = float(np.sum(g * o * (1.0 - o)))
    # Uncertainty = CRPS of the sample climatology of the observations
    uncertainty = float(np.abs(y_true[:, None] - y_true[None, :]).sum() / (2.0 * n_obs ** 2))
    resolution = uncertainty - crps_pot

    # -------------------------------
    # Per-bin table (unchanged): test rows sorted by predicted mean, split into n_bins groups
    # -------------------------------
    sort_idx = np.argsort(mu_pred)
    y_sorted = y_true[sort_idx]
    mu_sorted = mu_pred[sort_idx]
    sigma_sorted = sigma_pred[sort_idx]

    bins = np.array_split(np.arange(len(y_true)), n_bins)

    bin_results = []

    for i, b in enumerate(bins):
        bin_y = y_sorted[b]
        bin_mu = mu_sorted[b]
        bin_sigma = sigma_sorted[b]

        bin_crps = np.mean(
            ps.crps_gaussian(bin_y, mu=bin_mu, sig=bin_sigma)
        )

        bin_results.append({
            "Bin": i + 1,
            "Bin_Size": len(b),
            "Mean_Pred_Mu": np.mean(bin_mu),
            "Mean_Pred_Sigma": np.mean(bin_sigma),
            "Bin_CRPS": bin_crps
        })

    # -------------------------------
    # Summary table
    # -------------------------------
    summary_df = pd.DataFrame([{
        "Model": model_name,
        "Total_CRPS": total_crps,
        "Reliability": reliability,
        "Resolution": resolution,
        "Uncertainty": uncertainty,
        "Num_Bins": n_bins
    }])

    bin_df = pd.DataFrame(bin_results)

    # -------------------------------
    # Save to Excel
    # -------------------------------
    excel_path = Path(excel_path)
    excel_path.parent.mkdir(parents=True, exist_ok=True)

    sheet_summary = f"CRPS_Summary_{model_name}"
    sheet_bins = f"CRPS_Bins_{model_name}"

    with pd.ExcelWriter(_ensure_parent_dir(_ensure_excel_file(
        excel_path)),
        engine="openpyxl",
        mode="a" if excel_path.exists() else "w",
        if_sheet_exists="replace"
    ) as writer:
        writer
        summary_df.to_excel(writer, sheet_name=sheet_summary, index=False)
        writer
        bin_df.to_excel(writer, sheet_name=sheet_bins, index=False)

    print(f"✅ CRPS decomposition saved to:\n{excel_path}")
    print(f"   Sheets: {sheet_summary}, {sheet_bins}")

    return summary_df

In [15]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def generate_and_save_pit_results(
    predictions_df,
    y_true,
    excel_path,
    model_name="Model",
    n_bins=10
):
    """
    Generate PIT values, histogram, calibration diagnostics,
    and save everything to Excel with the PIT plot EMBEDDED.
    """

    # -------------------------------
    # Input handling
    # -------------------------------
    mu_pred = predictions_df['Mean'].values
    sigma_pred = predictions_df['StdDev'].values

    if 'Actual' in predictions_df.columns:
        y_true = predictions_df['Actual'].values
    else:
        y_true = np.asarray(y_true).ravel()

    y_true = np.asarray(y_true).ravel()

    sigma_pred = np.clip(sigma_pred, 1e-6, None)

    # -------------------------------
    # Compute PIT values
    # -------------------------------
    pit_values = norm.cdf(y_true, loc=mu_pred, scale=sigma_pred)

    pit_df = pd.DataFrame({"PIT_Value": pit_values})

    # -------------------------------
    # Histogram data
    # -------------------------------
    hist_density, bin_edges = np.histogram(
        pit_values, bins=n_bins, range=(0, 1), density=True
    )
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

    hist_df = pd.DataFrame({
        "Bin_Center": bin_centers,
        "Density": hist_density
    })

    # -------------------------------
    # Calibration diagnostics
    # -------------------------------
    pit_mean = pit_values.mean()
    pit_var = pit_values.var()
    ks_stat, ks_pvalue = kstest(pit_values, 'uniform')

    if ks_pvalue > 0.05:
        diagnosis = "Well-calibrated (Uniform PIT)"
    elif pit_var < 1/12:
        diagnosis = "Over-dispersed (Under-confident)"
    elif pit_var > 1/12:
        diagnosis = "Under-dispersed (Over-confident)"
    else:
        diagnosis = "Biased or miscalibrated"

    summary_df = pd.DataFrame([{
        "Model": model_name,
        "Mean_PIT": pit_mean,
        "Variance_PIT": pit_var,
        "Ideal_Variance_(1/12)": 1/12,
        "KS_Statistic": ks_stat,
        "KS_pvalue": ks_pvalue,
        "Calibration_Diagnosis": diagnosis
    }])

    # -------------------------------
    # Create PIT plot in memory
    # -------------------------------
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(
        pit_values,
        bins=n_bins,
        density=True,
        color="skyblue",
        edgecolor="black",
        alpha=0.75
    )
    ax.axhline(1, color="red", linestyle="--", label="Perfect Calibration")
    ax.set_xlabel("PIT Value")
    ax.set_ylabel("Density")
    ax.set_title(f"PIT Histogram – {model_name}")
    ax.legend()
    fig.tight_layout()

    img_buffer = BytesIO()
    fig.savefig(img_buffer, format="png", dpi=300)
    plt.close(fig)
    img_buffer.seek(0)

    # -------------------------------
    # Save to Excel
    # -------------------------------
    excel_path = Path(excel_path)
    excel_path.parent.mkdir(parents=True, exist_ok=True)

    sheet_values = f"PIT_Values_{model_name}"
    sheet_hist = f"PIT_Histogram_{model_name}"
    sheet_summary = f"PIT_Summary_{model_name}"
    sheet_plot = f"PIT_Plot_{model_name}"

    with pd.ExcelWriter(_ensure_parent_dir(_ensure_excel_file(
        excel_path)),
        engine="openpyxl",
        mode="a" if excel_path.exists() else "w",
        if_sheet_exists="replace"
    ) as writer:

        writer
        pit_df.to_excel(writer, sheet_name=sheet_values, index=False)
        writer
        hist_df.to_excel(writer, sheet_name=sheet_hist, index=False)
        writer
        summary_df.to_excel(writer, sheet_name=sheet_summary, index=False)

        # Create empty sheet for plot
        workbook = writer.book
        worksheet = workbook.create_sheet(sheet_plot)

        img = openpyxlImage(img_buffer)
        img.width = img.width * 0.4
        img.height = img.height * 0.4

        img.anchor = "B2"
        worksheet.add_image(img)

    print(f"✅ PIT values, histogram, summary, and EMBEDDED plot saved to:")
    print(f"   {excel_path}")

    return pit_df

In [16]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def generate_and_save_calibration_curve(
    predictions_df,
    y_true,
    excel_path,
    model_name="Model",
    n_quantiles=10
):
    """
    Generate quantile calibration curve (reliability diagram),
    compute calibration errors, and save everything to Excel
    with the plot EMBEDDED.
    """

    # -------------------------------
    # Input handling
    # -------------------------------
    mu_pred = predictions_df['Mean'].values
    sigma_pred = predictions_df['StdDev'].values

    if 'Actual' in predictions_df.columns:
        y_true = predictions_df['Actual'].values
    else:
        y_true = np.asarray(y_true).ravel()

    y_true = np.asarray(y_true).ravel()
    sigma_pred = np.clip(sigma_pred, 1e-6, None)

    # -------------------------------
    # Quantiles
    # -------------------------------
    quantiles = np.linspace(0, 1, n_quantiles + 1)

    observed_freq = []
    predicted_q = []

    for q in quantiles:
        # Predicted value at quantile q
        thresh = norm.ppf(q, loc=mu_pred, scale=sigma_pred)

        # Observed frequency
        obs = np.mean(y_true <= thresh)

        predicted_q.append(q)
        observed_freq.append(obs)

    predicted_q = np.array(predicted_q)
    observed_freq = np.array(observed_freq)

    # -------------------------------
    # Calibration metrics
    # -------------------------------
    mce = np.mean(np.abs(observed_freq - predicted_q))
    rmsce = np.sqrt(np.mean((observed_freq - predicted_q) ** 2))

    calib_df = pd.DataFrame({
        "Predicted_Quantile": predicted_q,
        "Observed_Frequency": observed_freq,
        "Absolute_Error": np.abs(observed_freq - predicted_q)
    })

    summary_df = pd.DataFrame([{
        "Model": model_name,
        "Mean_Calibration_Error (MCE)": mce,
        "RMS_Calibration_Error (RMSCE)": rmsce,
        "Num_Quantiles": n_quantiles
    }])

    # -------------------------------
    # Create calibration plot
    # -------------------------------
    fig, ax = plt.subplots(figsize=(7, 7))

    ax.plot([0, 1], [0, 1], 'k--', label="Perfect Calibration")
    ax.plot(
        predicted_q,
        observed_freq,
        'o-',
        color='teal',
        label=f"{model_name}"
    )

    ax.set_xlabel("Predicted Quantile (Confidence Level)")
    ax.set_ylabel("Observed Frequency")
    ax.set_title(f"Quantile Calibration Curve – {model_name}")
    ax.legend()
    fig.tight_layout()

    img_buffer = BytesIO()
    fig.savefig(img_buffer, format="png", dpi=300)
    plt.close(fig)
    img_buffer.seek(0)

    # -------------------------------
    # Save to Excel
    # -------------------------------
    excel_path = Path(excel_path)
    excel_path.parent.mkdir(parents=True, exist_ok=True)

    sheet_curve = f"Calibration_Curve_{model_name}"
    sheet_summary = f"Calibration_Summary_{model_name}"
    sheet_plot = f"Calibration_Plot_{model_name}"

    with pd.ExcelWriter(_ensure_parent_dir(_ensure_excel_file(
        excel_path)),
        engine="openpyxl",
        mode="a" if excel_path.exists() else "w",
        if_sheet_exists="replace"
    ) as writer:

        writer
        calib_df.to_excel(writer, sheet_name=sheet_curve, index=False)
        writer
        summary_df.to_excel(writer, sheet_name=sheet_summary, index=False)

        workbook = writer.book

        worksheet = workbook.create_sheet(sheet_plot)

        img = openpyxlImage(img_buffer)
        img.width = img.width * 0.4
        img.height = img.height * 0.4

        img.anchor = "B2"
        worksheet.add_image(img)

    print("✅ Calibration curve & metrics saved to Excel:")
    print(f"   {excel_path}")

    return calib_df

In [17]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
# Define the path to the folder
folder_path = "/content/drive/MyDrive/Regression/pile_uncertainty_analysis"

# Create the folder if it doesn't exist
os.makedirs(folder_path, exist_ok=True)

def plot_metrics_and_save_to_excel(writer, predictions_df, y_true, model_name):
    # Calculate metrics
    mean_pred = predictions_df['Mean']
    std_pred = predictions_df['StdDev']

    # Sharpness is the standard deviation
    sharpness = std_pred.mean()

    # CRPS for each prediction
    crps_values = ps.crps_gaussian(y_true, mean_pred, std_pred)

    # Negative Log Likelihood (NLL) for each prediction
    nll_values = -norm.logpdf(y_true, loc=mean_pred, scale=std_pred)
    nll = nll_values.mean()

    # Calibration (Validity)
    percentiles = np.linspace(0, 100, 101)
    calibration = [np.mean((y_true <= np.percentile(mean_pred + std_pred * np.random.randn(*mean_pred.shape), p))) for p in percentiles]

    # Create a DataFrame to store the metrics
    metrics_df = pd.DataFrame({
        'Mean Prediction': mean_pred,
        'Standard Deviation': std_pred,
        'CRPS': crps_values,
        'NLL': nll_values
    })

    # Add summary statistics
    summary_df = pd.DataFrame({
        'Metric': ['Sharpness', 'Mean CRPS', 'Mean NLL'],
        'Value': [sharpness, crps_values.mean(), nll]
    })

    # Write the metrics and summary to the Excel sheet
    metrics_sheet_name = f'{model_name} Metrics'
    writer
    metrics_df.to_excel(writer, sheet_name=metrics_sheet_name, index=False)

    summary_sheet_name = f'{model_name} Summary'
    writer
    summary_df.to_excel(writer, sheet_name=summary_sheet_name, index=False)

    # Plotting
    fig, axs = plt.subplots(2, 2, figsize=(12, 10))

    # Sharpness plot
    axs[0, 0].hist(std_pred, bins=30, color='skyblue', edgecolor='black')
    axs[0, 0].set_title(f'Sharpness (StdDev) for {model_name}')
    axs[0, 0].set_xlabel('Standard Deviation')
    axs[0, 0].set_ylabel('Frequency')

    # CRPS plot (distribution)
    axs[0, 1].hist(crps_values, bins=30, color='lightgreen', edgecolor='black')
    axs[0, 1].set_title(f'CRPS Distribution for {model_name}')
    axs[0, 1].set_xlabel('CRPS')
    axs[0, 1].set_ylabel('Frequency')

    # NLL plot (distribution)
    axs[1, 0].hist(nll_values, bins=30, color='salmon', edgecolor='black')
    axs[1, 0].set_title(f'NLL Distribution for {model_name}')
    axs[1, 0].set_xlabel('NLL')
    axs[1, 0].set_ylabel('Frequency')

    # Calibration plot
    axs[1, 1].plot(percentiles, calibration, label='Calibration Curve')
    axs[1, 1].plot(percentiles, percentiles / 100, linestyle='--', label='Perfect Calibration')
    axs[1, 1].set_title(f'Calibration (Validity) for {model_name}')
    axs[1, 1].set_xlabel('Predicted Percentile')
    axs[1, 1].set_ylabel('Observed Frequency')
    axs[1, 1].legend()

    plt.tight_layout()

    # Save plots to images
    plot_image_path = f'{model_name}_plots.png'
    fig.savefig(_ensure_parent_dir(plot_image_path))
    plt.close(fig)

    # Insert the plot image into the Excel sheet
    sheet_name = f'{model_name} Plots'
    if sheet_name not in writer.sheets:  # Check if sheet already exists
        # If sheet doesn't exist, create a new one
        worksheet = writer.book.add_worksheet(sheet_name)
    else:
        # If sheet exists, get the existing sheet object to avoid duplicates
        worksheet = writer.sheets[sheet_name]
    worksheet.insert_image('A1', plot_image_path)

# **Tuned Parameters**

In [18]:
# FIX K2: load the tuning results saved by hyperparameter_tuning/Optuna_autosampler.ipynb
# (was: a hand-pasted copy of an older tuning run that had been selected on the TEST set).
import pickle
if IN_KAGGLE:
    _bp = glob.glob("/kaggle/input/**/hyperparameter_tuning/best_params.pkl", recursive=True)[0]
else:
    _bp = "./drive/MyDrive/Regression/pile_uncertainty_analysis/hyperparameter_tuning/best_params.pkl"
with open(_bp, "rb") as f:
    best_scores_autosampler = pickle.load(f)
print("Loaded tuning results from", _bp, "-", len(best_scores_autosampler), "model x pruner entries")


Loaded tuning results from /kaggle/input/pile-demo-params/drive/MyDrive/Regression/pile_uncertainty_analysis/hyperparameter_tuning/best_params.pkl - 80 model x pruner entries


In [19]:
best_scores_autosampler

{('Random Forest', 'MedianPruner'): {'best_score': 0.14752680231770787,
  'best_params': {'n_estimators': 200,
   'criterion': 'absolute_error',
   'max_depth': 40,
   'min_samples_split': 2,
   'min_samples_leaf': 3,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 'sqrt',
   'max_leaf_nodes': 50,
   'min_impurity_decrease': 0.0,
   'n_jobs': -1,
   'random_state': 42,
   'verbose': 0,
   'warm_start': False,
   'ccp_alpha': 0.0},
  'test_mse': 0.14752680231770787,
  'test_rmse': 0.38409217945397933,
  'test_corr_coef': 0.9635104517818707,
  'pruner': 'MedianPruner'},
 ('Random Forest', 'NopPruner'): {'best_score': 0.1706761150095517,
  'best_params': {'n_estimators': 200,
   'criterion': 'squared_error',
   'max_depth': None,
   'min_samples_split': 2,
   'min_samples_leaf': 1,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 1.0,
   'max_leaf_nodes': 50,
   'min_impurity_decrease': 0.2,
   'n_jobs': -1,
   'random_state': 42,
   'verbose': 0,
   'warm_start': False,
   '

In [20]:
best_scores_autosampler

{('Random Forest', 'MedianPruner'): {'best_score': 0.14752680231770787,
  'best_params': {'n_estimators': 200,
   'criterion': 'absolute_error',
   'max_depth': 40,
   'min_samples_split': 2,
   'min_samples_leaf': 3,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 'sqrt',
   'max_leaf_nodes': 50,
   'min_impurity_decrease': 0.0,
   'n_jobs': -1,
   'random_state': 42,
   'verbose': 0,
   'warm_start': False,
   'ccp_alpha': 0.0},
  'test_mse': 0.14752680231770787,
  'test_rmse': 0.38409217945397933,
  'test_corr_coef': 0.9635104517818707,
  'pruner': 'MedianPruner'},
 ('Random Forest', 'NopPruner'): {'best_score': 0.1706761150095517,
  'best_params': {'n_estimators': 200,
   'criterion': 'squared_error',
   'max_depth': None,
   'min_samples_split': 2,
   'min_samples_leaf': 1,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 1.0,
   'max_leaf_nodes': 50,
   'min_impurity_decrease': 0.2,
   'n_jobs': -1,
   'random_state': 42,
   'verbose': 0,
   'warm_start': False,
   '

# **Probabilistic distributions using NGBoost**

In [21]:
# Assuming best_params is obtained correctly
best_params = get_best_model_params(best_scores_autosampler, 'NGBoost')

# Remove any conflicting parameters from best_params
incompatible_keys = ['Dist', 'Score']
for key in incompatible_keys:
    best_params.pop(key, None)

# Initialize NGBoost model with best_params
ngb_model = NGBRegressor(Dist=Normal, **best_params)

# Fit the model
ngb_model.fit(X_train, y_train)

# Predict the distribution
pred_dist = ngb_model.pred_dist(X_test)

# Extract mean and standard deviation
predictions_NGB_df = pd.DataFrame({
    'Mean': pred_dist.loc,
    'StdDev': pred_dist.scale,
    'Actual': y_test.ravel()
})

# Calculate and print the mean squared error
mse = mean_squared_error(y_test, pred_dist.loc)
print(f"Mean Squared Error: {mse:.2f}")

# Print the DataFrame with mean and standard deviation
print(predictions_NGB_df.head())

Mean Squared Error: 0.13
       Mean    StdDev  Actual
0  5.769538  0.041547    6.12
1  4.885089  0.907631    4.26
2  4.914554  0.972935    4.78
3  5.719123  0.038867    5.78
4  5.602269  0.038915    5.84


In [22]:
generate_and_save_plots(predictions_NGB_df, y_test, "./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx")

Excel file saved at ./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx


In [23]:
generate_and_save_crps_decomposition(
    predictions_NGB_df,
    y_test,
    "./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx",
    model_name="NGBoost",
    n_bins=10
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx
   Sheets: CRPS_Summary_NGBoost, CRPS_Bins_NGBoost


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,NGBoost,0.24087,0.028466,0.516613,0.72908,10


In [24]:
generate_and_save_pit_results(
    predictions_NGB_df,
    y_test,
    "./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx",
    model_name="NGBoost",
    n_bins=10
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx


,PIT_Value
0,1.000000e+00
1,2.455048e-01
2,4.450027e-01
3,9.413596e-01
4,1.000000e+00
5,4.208778e-02
6,3.689756e-01
7,1.134800e-02
8,7.421972e-01
9,1.727004e-01


In [25]:
generate_and_save_calibration_curve(
    predictions_NGB_df,
    y_test,
    "./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx",
    model_name="NGBoost",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.166667,0.066667
2,0.2,0.208333,0.008333
3,0.3,0.291667,0.008333
4,0.4,0.333333,0.066667
5,0.5,0.375000,0.125000
6,0.6,0.458333,0.141667
7,0.7,0.500000,0.200000
8,0.8,0.541667,0.258333
9,0.9,0.541667,0.358333


# **Probabilistic distributions using PGBM**

In [26]:
# Assuming best_params is obtained correctly
best_params = get_best_model_params(best_scores_autosampler, 'PGBM')

# Remove any conflicting parameters from best_params
# Note: Adjust the incompatible keys based on PGBM's requirements
incompatible_keys = ['Dist', 'Score']
for key in incompatible_keys:
    best_params.pop(key, None)

# Initialize PGBM model
pgbm_model = PGBM()

def mseloss_objective(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    gradient = yhat - y
    hessian = torch.ones_like(yhat)
    return gradient, hessian

def rmseloss_metric(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    loss = torch.sqrt(torch.mean((yhat - y) ** 2))
    return loss

X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)
# Fit the model
pgbm_model.train((X_train, y_train), objective=mseloss_objective, metric=rmseloss_metric, params = best_params)

# Predict the distribution
pred_dist = pgbm_model.predict_dist(X_test)

# Extract mean and standard deviation
# Note: PGBM might return predictions in a different format, adjust accordingly
predictions_PGBM_df = pd.DataFrame({
    'Mean': pred_dist.mean(axis=0),
    'StdDev': pred_dist.std(axis=0),
    'Actual': y_test.ravel()
})

# Calculate and print the mean squared error
mse = mean_squared_error(y_test, predictions_PGBM_df['Mean'])
print(f"Mean Squared Error: {mse:.2f}")

# Print the DataFrame with mean and standard deviation
print(predictions_PGBM_df.head())

Training on CPU


Estimator 0/100, Train metric: 1.2225
Estimator 1/100, Train metric: 1.0857
Estimator 2/100, Train metric: 0.9782
Estimator 3/100, Train metric: 0.8797
Estimator 4/100, Train metric: 0.7961
Estimator 5/100, Train metric: 0.7315
Estimator 6/100, Train metric: 0.6861
Estimator 7/100, Train metric: 0.6459
Estimator 8/100, Train metric: 0.6103
Estimator 9/100, Train metric: 0.5789
Estimator 10/100, Train metric: 0.5643
Estimator 11/100, Train metric: 0.5441
Estimator 12/100, Train metric: 0.5290
Estimator 13/100, Train metric: 0.5169
Estimator 14/100, Train metric: 0.5102
Estimator 15/100, Train metric: 0.5049
Estimator 16/100, Train metric: 0.5003
Estimator 17/100, Train metric: 0.4967
Estimator 18/100, Train metric: 0.4967
Estimator 19/100, Train metric: 0.4932
Estimator 20/100, Train metric: 0.4932
Estimator 21/100, Train metric: 0.4932
Estimator 22/100, Train metric: 0.4932
Estimator 23/100, Train metric: 0.4932
Estimator 24/100, Train metric: 0.4932
Estimator 25/100, Train metric: 0.4

In [27]:
generate_and_save_plots(predictions_PGBM_df, y_test, "./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx")

Excel file saved at ./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx


In [28]:
generate_and_save_crps_decomposition(
    predictions_PGBM_df,
    y_test,
    "./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx",
    model_name="PGBM",
    n_bins=10
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx
   Sheets: CRPS_Summary_PGBM, CRPS_Bins_PGBM


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,PGBM,0.299862,0.1078,0.536918,0.72908,10


In [29]:
generate_and_save_pit_results(
    predictions_PGBM_df,
    y_test,
    "./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx",
    model_name="PGBM",
    n_bins=10
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx


,PIT_Value
0,9.999999e-01
1,1.992851e-01
2,1.000000e+00
3,9.999976e-01
4,9.945073e-01
5,2.634710e-01
6,7.203230e-16
7,1.007115e-12
8,1.000000e+00
9,9.999999e-01


In [30]:
generate_and_save_calibration_curve(
    predictions_PGBM_df,
    y_test,
    "./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx",
    model_name="PGBM",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.250000,0.150000
2,0.2,0.291667,0.091667
3,0.3,0.333333,0.033333
4,0.4,0.333333,0.066667
5,0.5,0.333333,0.166667
6,0.6,0.333333,0.266667
7,0.7,0.416667,0.283333
8,0.8,0.458333,0.341667
9,0.9,0.500000,0.400000


# **Probabilistic Distribution using GPBoost (Treeffuser)**


In [31]:
best_params = get_best_model_params(best_scores_autosampler, 'GPBoost')

gpboost_score_engine = TreeBasedScoreModel(
    model_name='gpboost',
    model_params=best_params,
    seed=42,
    verbose=-1
)

# The fix is in the line below: '_score_model' has been corrected to 'score_model'.
model = Treeffuser(score_model=gpboost_score_engine, seed=42)
model.fit(X_train, y_train)

n_samples_for_prediction = 200
y_samples = model.sample(X_test, n_samples=n_samples_for_prediction, verbose=False)

y_samples_obj = Samples(y_samples)
mean = y_samples_obj.sample_mean()
stddev = y_samples_obj.sample_std()

predictions_GPBoost_TF_df = pd.DataFrame({
    'Mean': mean,
    'StdDev': stddev,
    'Actual': y_test
})

print(predictions_GPBoost_TF_df.head())


[1]	valid_0's l2: 0.521059
[2]	valid_0's l2: 0.51401
[3]	valid_0's l2: 0.50852
[4]	valid_0's l2: 0.503365
[5]	valid_0's l2: 0.498593
[6]	valid_0's l2: 0.49683
[7]	valid_0's l2: 0.493291
[8]	valid_0's l2: 0.491601
[9]	valid_0's l2: 0.486866
[10]	valid_0's l2: 0.481769
[11]	valid_0's l2: 0.47873
[12]	valid_0's l2: 0.473373
[13]	valid_0's l2: 0.468001
[14]	valid_0's l2: 0.462631
[15]	valid_0's l2: 0.461348
[16]	valid_0's l2: 0.459586
[17]	valid_0's l2: 0.457264
[18]	valid_0's l2: 0.454605
[19]	valid_0's l2: 0.453127
[20]	valid_0's l2: 0.452066
[21]	valid_0's l2: 0.450388
[22]	valid_0's l2: 0.445265
[23]	valid_0's l2: 0.443861
[24]	valid_0's l2: 0.439363
[25]	valid_0's l2: 0.434987
[26]	valid_0's l2: 0.434419
[27]	valid_0's l2: 0.431826
[28]	valid_0's l2: 0.432364
[29]	valid_0's l2: 0.428399
[30]	valid_0's l2: 0.426071
[31]	valid_0's l2: 0.422845
[32]	valid_0's l2: 0.419299
[33]	valid_0's l2: 0.414987
[34]	valid_0's l2: 0.412624
[35]	valid_0's l2: 0.410002
[36]	valid_0's l2: 0.409626
[37]	

       Mean     StdDev  Actual
0  7.091824   9.120469    6.12
1  3.746446   9.358511    4.26
2  3.968753  10.882888    4.78
3  7.112902   9.964337    5.78
4  6.869241  11.290561    5.84


In [32]:
generate_and_save_plots(predictions_GPBoost_TF_df, y_test, "./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx")

Excel file saved at ./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx


In [33]:
generate_and_save_crps_decomposition(
    predictions_GPBoost_TF_df,
    y_test,
    "./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx",
    model_name="GPBoost",
    n_bins=10
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx
   Sheets: CRPS_Summary_GPBoost, CRPS_Bins_GPBoost


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,GPBoost,2.438218,1.967846,0.258228,0.72908,10


In [34]:
generate_and_save_pit_results(
    predictions_GPBoost_TF_df,
    y_test,
    "./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx",
    model_name="GPBoost",
    n_bins=10
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx


,PIT_Value
0,0.457571
1,0.521881
2,0.529711
3,0.446793
4,0.463683
5,0.472747
6,0.489149
7,0.423582
8,0.433423
9,0.457709


In [35]:
generate_and_save_calibration_curve(
    predictions_GPBoost_TF_df,
    y_test,
    "./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx",
    model_name="GPBoost",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.000000,0.100000
2,0.2,0.000000,0.200000
3,0.3,0.000000,0.300000
4,0.4,0.000000,0.400000
5,0.5,0.833333,0.333333
6,0.6,1.000000,0.400000
7,0.7,1.000000,0.300000
8,0.8,1.000000,0.200000
9,0.9,1.000000,0.100000


# **Probabilistic Distribution using Gradient Boosting (Treeffuser)**

In [36]:
best_params = get_best_model_params(best_scores_autosampler, 'GradientBoosting')

gbr_score_engine = TreeBasedScoreModel(
    model_name='gradient_boosting',
    # FIX T1: the diffusion score model is fitted by least squares (Treeffuser's denoising
    # objective); the tuned GBR loss may be absolute_error/huber, so force squared_error here.
    model_params={**best_params, 'loss': 'squared_error'},
    seed=42,
)

model = Treeffuser(score_model=gbr_score_engine, seed=42)
model.fit(X_train, y_train)

n_samples_for_prediction = 200
y_samples = model.sample(X_test, n_samples=n_samples_for_prediction, verbose=False)

y_samples_obj = Samples(y_samples)
mean = y_samples_obj.sample_mean()
stddev = y_samples_obj.sample_std()

predictions_GBR_TF_df = pd.DataFrame({
    'Mean': mean,
    'StdDev': stddev,
    'Actual': y_test
})

print(predictions_GBR_TF_df.head())


       Mean     StdDev  Actual
0  7.247703   7.932730    6.12
1  5.705871  10.640964    4.26
2  4.488231  11.595935    4.78
3  7.277056   9.452541    5.78
4  6.800446   9.867998    5.84


In [37]:
generate_and_save_plots(predictions_GBR_TF_df, y_test, "./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx")

Excel file saved at ./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx


In [38]:
generate_and_save_crps_decomposition(
    predictions_GBR_TF_df,
    y_test,
    "./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx",
    model_name="Gradient Boosting",
    n_bins=10
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx
   Sheets: CRPS_Summary_Gradient Boosting, CRPS_Bins_Gradient Boosting


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,Gradient Boosting,2.52549,1.869333,0.072444,0.72908,10


In [39]:
generate_and_save_pit_results(
    predictions_GBR_TF_df,
    y_test,
    "./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx",
    model_name="GBoosting",
    n_bins=10
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx


,PIT_Value
0,0.443478
1,0.445959
2,0.510037
3,0.437080
4,0.461232
5,0.425569
6,0.473993
7,0.358705
8,0.396417
9,0.456273


In [40]:
generate_and_save_calibration_curve(
    predictions_GBR_TF_df,
    y_test,
    "./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx",
    model_name="GBoosting",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.000000,0.100000
2,0.2,0.000000,0.200000
3,0.3,0.000000,0.300000
4,0.4,0.083333,0.316667
5,0.5,0.833333,0.333333
6,0.6,1.000000,0.400000
7,0.7,1.000000,0.300000
8,0.8,1.000000,0.200000
9,0.9,1.000000,0.100000


# **Probabilistic Distribution using Catboost (Treeffuser)**

In [41]:
best_params = get_best_model_params(best_scores_autosampler, 'CatBoost')

catboost_score_engine = TreeBasedScoreModel(
    model_name='catboost',
    model_params=best_params,
    seed=42,
)

model = Treeffuser(score_model=catboost_score_engine, seed=42)
model.fit(X_train, y_train)

n_samples_for_prediction = 200
y_samples = model.sample(X_test, n_samples=n_samples_for_prediction, verbose=False)

y_samples_obj = Samples(y_samples)
mean = y_samples_obj.sample_mean()
stddev = y_samples_obj.sample_std()

predictions_CatBoost_TF_df = pd.DataFrame({
    'Mean': mean,
    'StdDev': stddev,
    'Actual': y_test
})

print(predictions_CatBoost_TF_df.head())

       Mean     StdDev  Actual
0  7.436371  15.483035    6.12
1  4.638020  26.593903    4.26
2  5.967961  18.383499    4.78
3  8.221480  14.102471    5.78
4  5.712393  13.467207    5.84


In [42]:
generate_and_save_plots(predictions_CatBoost_TF_df, y_test, "./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx")

Excel file saved at ./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx


In [43]:
generate_and_save_crps_decomposition(
    predictions_CatBoost_TF_df,
    y_test,
    "./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx",
    model_name="CatBoost",
    n_bins=10
)


✅ CRPS decomposition saved to:
drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx
   Sheets: CRPS_Summary_CatBoost, CRPS_Bins_CatBoost


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,CatBoost,4.625332,3.05161,-0.845542,0.72908,10


In [44]:
generate_and_save_pit_results(
    predictions_CatBoost_TF_df,
    y_test,
    "./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx",
    model_name="CatBoost",
    n_bins=10
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx


,PIT_Value
0,0.466123
1,0.494329
2,0.474238
3,0.431277
4,0.503780
5,0.518428
6,0.502299
7,0.353329
8,0.382396
9,0.430844


In [45]:
generate_and_save_calibration_curve(
    predictions_CatBoost_TF_df,
    y_test,
    "./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx",
    model_name="CatBoost",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.000000,0.100000
2,0.2,0.000000,0.200000
3,0.3,0.000000,0.300000
4,0.4,0.208333,0.191667
5,0.5,0.708333,0.208333
6,0.6,1.000000,0.400000
7,0.7,1.000000,0.300000
8,0.8,1.000000,0.200000
9,0.9,1.000000,0.100000


# **Probabilistic Distribution using XGBoost (Treeffuser)**

In [46]:
best_params = get_best_model_params(best_scores_autosampler, 'XGBoost')

xgb_score_engine = TreeBasedScoreModel(
    model_name='xgboost',
    model_params=best_params,
    seed=42,
    verbose=-1
)

model = Treeffuser(score_model=xgb_score_engine, seed=42)
model.fit(X_train, y_train)

n_samples_for_prediction = 200
y_samples = model.sample(X_test, n_samples=n_samples_for_prediction, verbose=False)

y_samples_obj = Samples(y_samples)
mean = y_samples_obj.sample_mean()
stddev = y_samples_obj.sample_std()

predictions_XGB_TF_df = pd.DataFrame({
    'Mean': mean,
    'StdDev': stddev,
    'Actual': y_test
})

print(predictions_XGB_TF_df.head())

[0]	validation_0-rmse:0.71994


[1]	validation_0-rmse:0.71741


[2]	validation_0-rmse:0.71140


[3]	validation_0-rmse:0.70926


[4]	validation_0-rmse:0.70226


[5]	validation_0-rmse:0.70399


... (291 outputs left out)

[297]	validation_0-rmse:0.82612


[298]	validation_0-rmse:0.82661


[299]	validation_0-rmse:0.82683

       Mean     StdDev  Actual
0  6.926609   6.001554    6.12
1 -0.054297  11.456816    4.26
2 -0.520064  10.611164    4.78
3  7.113227   6.974971    5.78
4  5.995621   5.343129    5.84


In [47]:
generate_and_save_plots(predictions_XGB_TF_df, y_test, "./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx")

Excel file saved at ./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx


In [48]:
generate_and_save_crps_decomposition(
    predictions_XGB_TF_df,
    y_test,
    "./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx",
    model_name="XGBoost",
    n_bins=10
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx
   Sheets: CRPS_Summary_XGBoost, CRPS_Bins_XGBoost


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,XGBoost,1.721399,0.669027,-0.323583,0.72908,10


In [49]:
generate_and_save_pit_results(
    predictions_XGB_TF_df,
    y_test,
    "./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx",
    model_name="XGBoost",
    n_bins=10
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx


,PIT_Value
0,0.446543
1,0.646754
2,0.691279
3,0.424206
4,0.488382
5,0.453725
6,0.500032
7,0.294820
8,0.367829
9,0.536695


In [50]:
generate_and_save_calibration_curve(
    predictions_XGB_TF_df,
    y_test,
    "./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx",
    model_name="XGBoost",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.000000,0.100000
2,0.2,0.000000,0.200000
3,0.3,0.041667,0.258333
4,0.4,0.125000,0.275000
5,0.5,0.541667,0.041667
6,0.6,0.750000,0.150000
7,0.7,1.000000,0.300000
8,0.8,1.000000,0.200000
9,0.9,1.000000,0.100000


# **Probabilistic Distribution using LGBM (Treeffuser)**

In [51]:
best_params = get_best_model_params(best_scores_autosampler, 'LightGBM')

lgbm_score_engine = TreeBasedScoreModel(
    model_name='lightgbm',
    model_params=best_params,
    seed=42,
    verbose=-1
)

model = Treeffuser(score_model=lgbm_score_engine, seed=42)
model.fit(X_train, y_train)

n_samples_for_prediction = 200
y_samples = model.sample(X_test, n_samples=n_samples_for_prediction, verbose=False)

y_samples_obj = Samples(y_samples)
mean = y_samples_obj.sample_mean()
stddev = y_samples_obj.sample_std()

predictions_LGBM_TF_df = pd.DataFrame({
    'Mean': mean,
    'StdDev': stddev,
    'Actual': y_test
})

print(predictions_LGBM_TF_df.head())


       Mean     StdDev  Actual
0  7.119270  14.920980    6.12
1  2.006222  13.507525    4.26
2  1.961672  10.845548    4.78
3  8.753924   9.877894    5.78
4  5.853214   7.302716    5.84


In [52]:
generate_and_save_plots(predictions_LGBM_TF_df, y_test, "./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx")

Excel file saved at ./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx


In [53]:
generate_and_save_crps_decomposition(
    predictions_LGBM_TF_df,
    y_test,
    "./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx",
    model_name="LightGBM",
    n_bins=10
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx
   Sheets: CRPS_Summary_LightGBM, CRPS_Bins_LightGBM


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,LightGBM,2.640833,1.591884,-0.320409,0.72908,10


In [54]:
generate_and_save_pit_results(
    predictions_LGBM_TF_df,
    y_test,
    "./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx",
    model_name="LightGBM",
    n_bins=10
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx


,PIT_Value
0,0.473302
1,0.566257
2,0.602514
3,0.381681
4,0.499278
5,0.421120
6,0.554342
7,0.382854
8,0.399699
9,0.424285


In [55]:
generate_and_save_calibration_curve(
    predictions_LGBM_TF_df,
    y_test,
    "./drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx",
    model_name="LightGBM",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.000000,0.100000
2,0.2,0.000000,0.200000
3,0.3,0.000000,0.300000
4,0.4,0.125000,0.275000
5,0.5,0.625000,0.125000
6,0.6,0.916667,0.316667
7,0.7,1.000000,0.300000
8,0.8,1.000000,0.200000
9,0.9,1.000000,0.100000


# **IBUG Installation**

In [56]:
# Clone the repository
!git clone https://github.com/jjbrophy47/ibug.git
%cd ibug

# Build extensions if needed
!python setup.py build_ext --inplace

# Install the package
!pip install .

# Verify installation
try:
    from ibug import IBUGWrapper
    print("IBUGWrapper imported successfully!")
except ImportError as e:
    print("Error importing IBUGWrapper:", e)

Cloning into 'ibug'...


remote: Enumerating objects: 1763, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (31/31), done.


remote: Total 1763 (delta 18), reused 15 (delta 4), pack-reused 1726 (from 1)
Receiving objects: 100% (1763/1763), 1.31 MiB | 4.52 MiB/s, done.


Resolving deltas: 100% (967/967), done.


/kaggle/working/ibug


Compiling ibug/parsers/_tree32.pyx because it changed.
Compiling ibug/parsers/_tree64.pyx because it changed.
[1/2] Cythonizing ibug/parsers/_tree32.pyx


... (16 outputs left out)

  Preparing metadata (pyproject.toml) ... done


  Created wheel for ibug: filename=ibug-0.0.12-cp313-cp313-linux_x86_64.whl size=1581580 sha256=dda081ecd67d08a75a3caa287e8e2b00ae30d02ac64194c9dbd126460b67650d
  Stored in directory: /tmp/pip-ephem-wheel-cache-0f3j26yy/wheels/48/c2/c1/d39bc91440bc7e7da40b146580d8aeb84c5361fa857fe8037e
Successfully built ibug


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [ibug]


IBUGWrapper imported successfully!


In [57]:
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.3, random_state=42)  # FIX P2: seeded


# **Probabilistic Distributions using LGBM (IBUG)**

In [58]:
# y_train = y_train.to_numpy()
# y_val = y_val.to_numpy()

In [59]:
# Assume hyperparameter tuning has been done and best_params are obtained
best_params = get_best_model_params(best_scores_autosampler, 'LightGBM')

# --- Ensure IBUG compatibility ---
best_params['reg_alpha'] = 0
best_params['class_weight'] = None
best_params['boosting_type'] = 'gbdt'

# Initialize and train the LightGBM regressor using the best_params
model = LGBMRegressor(**best_params)
model.fit(X_train, y_train)

# Extend the trained model into a probabilistic predictor using IBUGWrapper
prob_model = IBUGWrapper().fit(model, X_train, y_train, X_val=X_val, y_val=y_val)

# Use the model to predict the probabilistic distribution on the test data.
location, scale = prob_model.pred_dist(X_test)

# Optionally, to also get the k-nearest training indices and their values used in modeling
location2, scale2, train_idxs, train_vals = prob_model.pred_dist(X_test, return_kneighbors=True)

dev = scale  # FIX P2: IBUG's scale is already a standard deviation (np.std of neighbours); was scale**0.5
# Create a DataFrame for clarity
predictions_LGBM_df = pd.DataFrame({
    'Mean': location,
    'StdDev': dev,
    'Actual': y_test,
})

print(predictions_LGBM_df.head())

       Mean    StdDev  Actual
0  5.859101  0.416161    6.12
1  4.287214  1.014043    4.26
2  4.498250  0.985385    4.78
3  5.676704  0.303776    5.78
4  5.786139  0.303776    5.84


In [60]:
%cd ..

/kaggle/working


In [61]:
%cd ..

/kaggle


In [62]:
generate_and_save_plots(predictions_LGBM_df, y_test, "./content/drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx")

Excel file saved at ./content/drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx


In [63]:
generate_and_save_crps_decomposition(
    predictions_LGBM_df,
    y_test,
    "/content/drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx",
    model_name="LGBM",
    n_bins=10
)

✅ CRPS decomposition saved to:
/content/drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx
   Sheets: CRPS_Summary_LGBM, CRPS_Bins_LGBM


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,LGBM,0.276144,0.032004,0.484903,0.72908,10


In [64]:
generate_and_save_pit_results(
    predictions_LGBM_df,
    y_test,
    "/content/drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx",
    model_name="LGBM",
    n_bins=10
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   /content/drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx


,PIT_Value
0,0.734644
1,0.489295
2,0.612534
3,0.633087
4,0.570365
5,0.309636
6,0.331224
7,0.340618
8,0.704175
9,0.881421


In [65]:
generate_and_save_calibration_curve(
    predictions_LGBM_df,
    y_test,
    "/content/drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx",
    model_name="LGBM",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   /content/drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.000000,0.100000
2,0.2,0.000000,0.200000
3,0.3,0.125000,0.175000
4,0.4,0.291667,0.108333
5,0.5,0.416667,0.083333
6,0.6,0.583333,0.016667
7,0.7,0.750000,0.050000
8,0.8,0.875000,0.075000
9,0.9,1.000000,0.100000


# **Probabilistic Distributions using XGBoost (IBUG)**

In [66]:
# Assume hyperparameter tuning has been done and best_params are obtained
best_params = get_best_model_params(best_scores_autosampler, 'XGBoost')

best_params['reg_alpha'] = 0
best_params['scale_pos_weight'] = 1
best_params['base_score'] = 0.5

# Initialize and train the XGBoost regressor using the best_params
model = XGBRegressor(**best_params)
model.fit(X_train, y_train)

# Extend the trained model into a probabilistic predictor using IBUGWrapper
prob_model = IBUGWrapper().fit(model, X_train, y_train, X_val=X_val, y_val=y_val)

# Use the model to predict the probabilistic distribution on the test data.
# This returns the predicted mean (location) and uncertainty (scale, variance proxy)
location, scale = prob_model.pred_dist(X_test)

# Optionally, to also get the k-nearest training indices and their values used in modeling
location, scale, train_idxs, train_vals = prob_model.pred_dist(X_test, return_kneighbors=True)

# Calculate the standard deviation from the variance (scale)
std_dev = scale  # FIX P2: IBUG's scale is already a standard deviation (np.std of neighbours); was scale ** 0.5

# Create a DataFrame for clarity
predictions_XGB_df = pd.DataFrame({
    'Mean': location,
    'StdDev': std_dev,
    'Actual': y_test
})

print(predictions_XGB_df.head())

       Mean    StdDev  Actual
0  5.982838  0.306532    6.12
1  4.459758  0.959770    4.26
2  4.383588  0.956135    4.78
3  5.401042  0.715237    5.78
4  5.842930  0.573371    5.84


In [67]:
generate_and_save_plots(predictions_XGB_df, y_test, "./content/drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx")  # FIX P3: was predictions_LGBM_df

Excel file saved at ./content/drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx


In [68]:
generate_and_save_crps_decomposition(
    predictions_XGB_df,
    y_test,
    "/content/drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx",
    model_name="XGB",
    n_bins=10
)

✅ CRPS decomposition saved to:
/content/drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx
   Sheets: CRPS_Summary_XGB, CRPS_Bins_XGB


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,XGB,0.266257,0.046862,0.509651,0.72908,10


In [69]:
generate_and_save_pit_results(
    predictions_XGB_df,
    y_test,
    "/content/drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx",
    model_name="XGB",
    n_bins=10
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   /content/drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx


,PIT_Value
0,0.672730
1,0.417563
2,0.660782
3,0.701887
4,0.497961
5,0.484558
6,0.369841
7,0.382569
8,0.730452
9,0.583729


In [70]:
generate_and_save_calibration_curve(
    predictions_XGB_df,
    y_test,
    "/content/drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx",
    model_name="XGB",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   /content/drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.083333,0.016667
2,0.2,0.083333,0.116667
3,0.3,0.083333,0.216667
4,0.4,0.250000,0.150000
5,0.5,0.416667,0.083333
6,0.6,0.583333,0.016667
7,0.7,0.666667,0.033333
8,0.8,0.833333,0.033333
9,0.9,0.958333,0.058333


# **Matrix Evaulation**

In [71]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

# Create a Pandas Excel writer using XlsxWriter as the engine.
folder_path = "/content/drive/MyDrive/Regression/pile_uncertainty_analysis"
excel_path = os.path.join(folder_path, "/content/drive/MyDrive/Regression/pile_uncertainty_analysis/probabilistic_distribution/Matrix Evaluation.xlsx")
with pd.ExcelWriter(_ensure_parent_dir(excel_path), engine='xlsxwriter') as writer:
    plot_metrics_and_save_to_excel(writer, predictions_NGB_df, y_test, 'NGBoost')
    plot_metrics_and_save_to_excel(writer, predictions_PGBM_df, y_test, 'PGBM')
    plot_metrics_and_save_to_excel(writer, predictions_LGBM_df, y_test, 'LGBM')
    plot_metrics_and_save_to_excel(writer, predictions_XGB_df, y_test, 'XGBoost')
    plot_metrics_and_save_to_excel(writer, predictions_LGBM_TF_df, y_test, "LightGBM_TF")
    plot_metrics_and_save_to_excel(writer, predictions_XGB_TF_df, y_test, "XGBoost_TF")
    plot_metrics_and_save_to_excel(writer, predictions_CatBoost_TF_df, y_test, "CatBoost_TF")
    plot_metrics_and_save_to_excel(writer, predictions_GPBoost_TF_df, y_test, "GPBoost_TF")
    plot_metrics_and_save_to_excel(writer, predictions_GBR_TF_df, y_test, "GradientBoosting_TF")